In [18]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import cvxpy as cp
import warnings
warnings.filterwarnings('ignore')

In [19]:
COLUMNS = {
    'DY-ADJ_AF-OPEN_PRICE_2': 'open',
    'DY-ADJ_AF-HIGHEST_PRICE_2': 'high',
    'DY-ADJ_AF-LOWEST_PRICE_2': 'low',
    'DY-ADJ_AF-CLOSE_PRICE_2': 'close',
    'DY-BASIC-MARKET_VALUE': 'market_cap',

    'LEVEL1_NAME': 'industry',
    'LEVEL2_NAME': 'sub_industry',
    'LEVEL3_NAME': 'sub_sub_industry',
}

In [27]:
def load_data(file_path: str, columns: list[str] = None, rename_map: dict[str, str] = COLUMNS) -> pd.DataFrame:
    """
    载入数据
    """
    df = pd.read_parquet(file_path, columns=columns)

    if file_path == '停牌.parquet':
        df = df.pivot(index='日期', columns='股票代码', values='是否停牌').fillna(0).astype(bool)
        df = df.stack().to_frame('suspended')
    elif file_path == 'st.parquet':
        df = df.notna().fillna(False).astype(bool)
        df = df.stack().to_frame('st')
    elif file_path == 'industry.parquet':
        df = df.swaplevel('CODE', 'DATE')
        df = df.sort_index()

    df.index = df.index.set_names(['date', 'ticker'])
    df = df.rename(columns=rename_map)
    return df

In [21]:
def calculate_factor_and_returns(df: pd.DataFrame) -> pd.DataFrame:
    """
    计算因子值和未来收益率，并剔除不符合条件的股票
    """
    df = df.sort_values(['CODE', 'DATE'])

    df['factor'] = df
    
    return df

In [28]:
daily_data = load_data('daily_data.parquet', ['DATE', 'CODE', 'DY-ADJ_AF-OPEN_PRICE_2', 'DY-ADJ_AF-HIGHEST_PRICE_2',
                                                    'DY-ADJ_AF-LOWEST_PRICE_2', 'DY-ADJ_AF-CLOSE_PRICE_2',
                                                    'DY-BASIC-MARKET_VALUE'])
industry_data = load_data('industry.parquet', ['CODE', 'LEVEL1_NAME', 'LEVEL2_NAME', 'LEVEL3_NAME'])
st_data = load_data('st.parquet')
tp_data = load_data('停牌.parquet')

In [29]:
daily_data

open      high       low     close    market_cap
date       ticker                                                      
2010-01-04 000001  1011.712  1014.188   977.053   978.291  7.362983e+10
           000002  1099.530  1101.557  1074.195  1074.195  1.165492e+11
           000004     0.000     0.000     0.000    68.339  8.397668e+08
           000005    59.055    59.448    58.072    58.858  5.476858e+09
           000006   227.094   227.495   222.685   222.885  5.639878e+09
...                     ...       ...       ...       ...           ...
2021-12-31 688799    40.020    41.720    40.020    41.170  3.861746e+09
           688800   145.370   149.000   137.500   139.280  1.504224e+10
           688819    43.126    43.512    43.014    43.430  4.160588e+10
           688981    52.900    53.450    52.740    52.990  4.188254e+11
           689009    69.000    71.250    67.300    70.070  4.960164e+10

[8717164 rows x 5 columns]

In [30]:
industry_data

industry sub_industry sub_sub_industry
date     ticker                                       
20100104 000001     金融服务           银行               银行
         000002      房地产        房地产开发             住宅开发
         000003       综合           综合               综合
         000004     医药生物         生物制品             生物制品
         000005      房地产        房地产开发             住宅开发
...                  ...          ...              ...
20211231 688799     医药生物         化学制药             化学制剂
         688800       电子         其他电子             其他电子
         688819     电力设备           电池         蓄电池及其他电池
         688981       电子          半导体           集成电路制造
         689009       汽车       摩托车及其他           其他运输设备

[9012858 rows x 3 columns]

In [31]:
st_data

st
date     ticker       
20100104 000001  False
         000002  False
         000003   True
         000004   True
         000005  False
...                ...
20221230 688799  False
         688800  False
         688819  False
         688981  False
         689009  False

[16963830 rows x 1 columns]

In [32]:
tp_data

suspended
date       ticker           
2010-01-04 000001      False
           000002      False
           000004       True
           000005      False
           000006      False
...                      ...
2021-12-31 871396      False
           871553      False
           871642      False
           871981      False
           872925      False

[13893671 rows x 1 columns]